<a href="https://colab.research.google.com/github/pradh/tools/blob/ml/dc-embed/SEARCH_StatVarEmbeddings_Basic.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [64]:
BUILDS = ['demographics300', 'uncurated3000']

## Load model, copy over and load embeddings


In [29]:
%%capture
from google.colab import auth
auth.authenticate_user()

!gcloud config set project 'datcom-204919'
!gsutil -m cp gs://datcom-csv/embeddings/embeddings_*.csv .

In [32]:
!ls -l

total 18164
-rw-r--r-- 1 root root  2187591 Dec 16 07:47 embeddings_demographics300.csv
-rw-r--r-- 1 root root 16402373 Dec 16 07:47 embeddings_uncurated3000.csv
drwxr-xr-x 1 root root     4096 Dec 16 00:01 sample_data


In [31]:
%%capture
!pip install -U sentence-transformers
!pip install datasets

In [33]:
from sentence_transformers import SentenceTransformer, util

# Download model
model = SentenceTransformer('all-MiniLM-L6-v2')

import torch
from datasets import load_dataset

# Load for Search below
dataset_embeddings_maps = {}
dcid_maps = {}
for build in BUILDS:
  print('Loading build ', build)
  ds = load_dataset('csv', data_files=f'embeddings_{build}.csv')

  df = ds["train"].to_pandas()
  dcid_maps[build] = df['dcid'].values.tolist()
  df = df.drop('dcid', axis=1)

  dataset_embeddings_maps[build] = torch.from_numpy(df.to_numpy()).to(torch.float)

Loading build  demographics300


Extracting data files:   0%|          | 0/1 [00:00<?, ?it/s]

Generating train split: 0 examples [00:00, ? examples/s]

Dataset csv downloaded and prepared to /root/.cache/huggingface/datasets/csv/default-331cbe5df9fe1d75/0.0.0/6b34fb8fcf56f7c8ba51dc895bfa2bfbe43546f190a60fcf74bb5e8afdcc2317. Subsequent calls will reuse this data.


  0%|          | 0/1 [00:00<?, ?it/s]

Loading build  uncurated3000


Extracting data files:   0%|          | 0/1 [00:00<?, ?it/s]

Generating train split: 0 examples [00:00, ? examples/s]

Dataset csv downloaded and prepared to /root/.cache/huggingface/datasets/csv/default-a4edccfb3af412ef/0.0.0/6b34fb8fcf56f7c8ba51dc895bfa2bfbe43546f190a60fcf74bb5e8afdcc2317. Subsequent calls will reuse this data.


  0%|          | 0/1 [00:00<?, ?it/s]

## 2. Search


In [86]:
#@title { run: "auto", vertical-output: true }

BUILD = 'demographics300'  #@param ['demographics300', 'uncurated3000']
QUERY = "cannot afford stuff " #@param {type:"string"}

query_embeddings = model.encode([QUERY])

from sentence_transformers.util import semantic_search
hits = semantic_search(query_embeddings, dataset_embeddings_maps[BUILD], top_k=15)

# Note: multiple results may map to the same DCID. As well, the same string may
# map to multiple DCIDs with the same score.
sv2score = {}
score2svs = {}
for e in hits[0]:
  for d in dcid_maps[BUILD][e['corpus_id']].split(','):
    s = e['score']
    # Prefer the top score.
    if d not in sv2score:
      sv2score[d] = s
      if s not in score2svs:
        score2svs[s] = [d]
      else:
        score2svs[s].append(d)

# Sort by scores
scores = [s for s in sorted(score2svs.keys(), reverse=True)]
svs = [' : '.join(score2svs[s]) for s in scores]

# Addd to Pandas
import pandas as pd
result = pd.DataFrame({'SV': svs, 'Cosine Score': scores})
pd.set_option('max_colwidth', 400)
result

,SV,Cosine Score
0,UnemploymentRate_Person,0.403417
1,Count_Person_Urban_BelowPovertyLevelInThePast12Months,0.388378
2,UrbanPovertyLine,0.369730
3,Count_Person_BelowPovertyLevelInThePast12Months,0.362357
4,Count_Person_BelowPovertyLevelInThePast12Months_BlackOrAfricanAmericanAlone,0.358177
5,Count_Person_BelowPovertyLevelInThePast12Months_AsFractionOf_Count_Person,0.352329
6,Count_Person_Urban_BelowPovertyLevelInThePast12Months_AsFractionOf_Count_Person_Urban,0.351113
7,Count_Person_Female_BelowPovertyLevelInThePast12Months,0.346928
8,Count_Person_AbovePovertyLevelInThePast12Months,0.346239
9,Count_Person_AbovePovertyLevelInThePast12Months_BlackOrAfricanAmericanAlone,0.344233
